# Modir — Whisper Arabic ASR fine-tune (Colab GPU)

**Code-first (AD-12.1):** the pipeline lives in `app/asr/*.py`; this notebook only RUNS
it on a GPU. You upload the code as a zip (no git clone / GitHub token needed), train,
then download the outputs back into the repo. No training logic lives here.

**Dataset:** Google FLEURS `ar_eg` — official, public, Parquet (no gating, no loading
script). Read Modern Standard Arabic, **NOT** Lebanese dialect — the honesty caveat
(AD-12.5) holds: we prove the pipeline + a real WER on public Arabic.

Runtime → Change runtime type → **GPU** (T4 is fine) before running.

## 1. Upload the code + install deps

First, in VS Code: `cd backend && tar -czf asr_code.tgz app`

Then run the cell below and pick `asr_code.tgz`. The heavy torch/transformers/datasets
stack installs ONLY here (AD-12.6), plus the two base deps `app.asr` needs (it uses the
project's structlog logger: `app.infra.logging` → `app.infra.settings` → pydantic-settings).

In [ ]:
import os
import subprocess

from google.colab import files

# Upload backend/app as asr_code.tgz (made locally: cd backend && tar -czf asr_code.tgz app).
# Colab does NOT overwrite a re-upload — it renames to "asr_code (2).tgz" etc. — so we
# extract the FILE THAT WAS JUST RETURNED, not a hard-coded name. (Avoids running stale code.)
uploaded = files.upload()
tgz = list(uploaded)[-1]
print("extracting:", tgz)
subprocess.run(["rm", "-rf", "/content/app"], check=False)
subprocess.run(["tar", "-xzf", tgz, "-C", "/content/"], check=True)
os.chdir("/content")
os.environ["PYTHONPATH"] = "/content"

# FLEURS is Parquet, so datasets has no script/trust_remote_code issue; any modern version
# works (pinned <4 only to match the repo's pyproject asr extra).
subprocess.run(
    [
        "pip", "install", "-q",
        "torch>=2.5.0", "transformers>=4.46.0,<5.0", "datasets>=3.1.0,<4.0",
        "evaluate>=0.4.3", "jiwer>=3.0.5", "soundfile>=0.12.1", "accelerate>=1.1.0",
        "structlog>=24.4.0", "pydantic-settings>=2.6.0",
    ],
    check=True,
)
# Confirm we are running the just-uploaded code, not a stale copy.
subprocess.run(["grep", "-n", "gradient_checkpointing", "/content/app/asr/train.py"], check=False)
print("ready:", os.path.isfile("/content/app/asr/train.py"))

## 2. Run the fine-tune

Smoke first (tiny subset, 2 steps, ~1 min) to prove the whole
train→eval→artifact→card→results.csv path; then the real run. FLEURS is public, so there is
NO HF login / dataset-terms step.

The real run writes the model + processor to `app/asr/artifacts/whisper-small-ar`, the
committed `model_card.json`, and appends the zero-shot + fine-tuned WER/CER rows to
`app/asr/results.csv`.

In [ ]:
# Smoke: validate the pipeline end to end (cd + PYTHONPATH inline so app.asr resolves).
!cd /content && PYTHONPATH=/content python -m app.asr.train --smoke

# Real fine-tune — comment the smoke line above, uncomment this:
# !cd /content && PYTHONPATH=/content python -m app.asr.train --epochs 3

## 3. Download the outputs back to the repo

- `results.csv` + `model_card.json` are SMALL and COMMITTED → put them in `backend/app/asr/`.
- The fine-tuned model (~1 GB) is git-ignored (AD-12.4) → put it in
  `backend/app/asr/artifacts/whisper-small-ar/` (or push to HF/Drive and set `WHISPER_MODEL_URI`).

For the **smoke** run you can skip the artifact entirely — just confirm `results.csv` has 2 rows.

In [ ]:
from google.colab import files

files.download("/content/app/asr/results.csv")      # → backend/app/asr/ (committed)
files.download("/content/app/asr/model_card.json")  # → backend/app/asr/ (committed)

# Real-run artifact (~1 GB; browser download is slow — Drive/HF is better):
# import subprocess
# subprocess.run(
#     ["tar", "-czf", "/content/whisper-small-ar.tgz",
#      "-C", "/content/app/asr/artifacts", "whisper-small-ar"],
#     check=True,
# )
# files.download("/content/whisper-small-ar.tgz")  # → backend/app/asr/artifacts/ (git-ignored)